# 00. 환경 설정 및 패키지 설치

**실행 순서:** 이 노트북을 가장 먼저 실행하세요.  
패키지 설치 후 **커널을 재시작**해야 합니다.

### 프로젝트 구조
```
C:\배그분석\
├── config.py                ← 공통 설정 (BASE_DIR, 날짜, 경로 등) — 여기서만 수정
├── 00_setup.ipynb           ← 지금 이 파일 (패키지 설치 + 경로 확인)
├── 01_data_pipeline.ipynb   ← 원시 telemetry → 피처 테이블 생성
├── 02_clustering.ipynb      ← 에란겔 페르소나 스코어링 (Robust Z-Score 규칙 배정)
├── 03_apply_maps.ipynb      ← 타 맵 적용 및 맵 간 전략 비교
├── 04_visualization.ipynb   ← 인터랙티브 시각화 대시보드
└── analysis_output\         ← 파이프라인 출력 (parquet, png, html)
    ├── erangel_features.parquet    ← 단일 진실 테이블 (01 생성)
    ├── erangel_clustered.parquet   ← 세그먼트 포함 (02 생성)
    ├── miramar/taego/rondo_clustered.parquet  (03 생성)
    └── models\erangel_model.pkl   ← fit_stats 저장 (02 생성)
```

> 설정 변경 시: `config.py`만 수정하면 01~04 전체에 반영됩니다.  



In [1]:
# 패키지 설치 (hdbscan은 Python 3.14 미지원 → sklearn 내장 사용)
import subprocess, sys

packages = ['umap-learn', 'plotly', 'nbformat', 'tqdm']

for pkg in packages:
    print(f'Installing {pkg}')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f' {pkg} 설치 완료')
    else:
        print(f'  FAIL {pkg} 설치 실패')
        print(result.stderr[:300])


Installing umap-learn
 umap-learn 설치 완료
Installing plotly
 plotly 설치 완료
Installing nbformat
 nbformat 설치 완료
Installing tqdm
 tqdm 설치 완료


In [2]:
# 설치 후 반드시 커널 재시작!
# VSCode: 상단 메뉴 -> Restart Kernel
print('위 셀 실행 후 커널을 재시작하세요!')
print('재시작 후 아래 셀을 실행해 설치를 확인하세요.')


위 셀 실행 후 커널을 재시작하세요!
재시작 후 아래 셀을 실행해 설치를 확인하세요.


In [3]:
# 커널 재시작 후 실행 - 설치 확인
import importlib

required = {
    'duckdb':   'duckdb',
    'umap':     'umap-learn',
    'sklearn':  'scikit-learn',
    'plotly':   'plotly',
    'pandas':   'pandas',
    'numpy':    'numpy',
    'tqdm':     'tqdm',
}

all_ok = True
for module, pkg in required.items():
    try:
        m = importlib.import_module(module)
        ver = getattr(m, '__version__', 'unknown')
        print(f' {pkg:<20} v{ver}')
    except ImportError:
        print(f' {pkg:<20} 설치 필요')
        all_ok = False

# sklearn HDBSCAN 확인
try:
    from sklearn.cluster import HDBSCAN
    print(' sklearn.cluster.HDBSCAN  내장 확인')
except ImportError:
    print(' sklearn HDBSCAN 없음 - sklearn 버전 확인 필요')
    all_ok = False

print()
if all_ok:
    print('모든 패키지 준비 완료 01_data_pipeline.ipynb 로 이동')
else:
    print('누락 패키지를 설치 후 다시 확인')


 duckdb               v1.4.4


c:\Users\dpqms\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 umap-learn           v0.5.11
 scikit-learn         v1.8.0
 plotly               v6.5.2
 pandas               v3.0.0
 numpy                v2.4.1
 tqdm                 v4.67.3
 sklearn.cluster.HDBSCAN  내장 확인

모든 패키지 준비 완료 01_data_pipeline.ipynb 로 이동


In [2]:
# 데이터 경로 확인 + config.py 존재 여부 검증
import glob, os

BASE_DIR = r'C:\배그분석'  # config.py와 동일하게 맞출 것

# config.py 존재 확인
config_path = os.path.join(BASE_DIR, 'config.py')
if os.path.exists(config_path):
    print(f' config.py 확인: {config_path}')
else:
    print(f'  config.py 없음: {config_path}')
    print('   → config.py를 BASE_DIR에 복사하세요')
    print('   → 모든 노트북이 이 파일에서 설정을 읽습니다')

# analysis_output 폴더 확인
output_dir = os.path.join(BASE_DIR, 'analysis_output')
print(f'\nanalysis_output 폴더: {"존재" if os.path.exists(output_dir) else " 없음 (01 실행 시 자동 생성)"}')

# telemetry parquet 파일
TELEMETRY_GLOB = BASE_DIR + r'\temp_parquet_files_*\Erangel_*.parquet'
MATCHES_GLOB   = BASE_DIR + r'\matches_*.csv'

tel_files = sorted(glob.glob(TELEMETRY_GLOB))
mat_files = sorted(glob.glob(MATCHES_GLOB))

print(f'\n텔레메트리 parquet: {len(tel_files)}개')
for f in tel_files[:5]:
    print(f'  {f}')
if len(tel_files) > 5:
    print(f'  외 {len(tel_files)-5}개')

print(f'\nmatches CSV: {len(mat_files)}개')
for f in mat_files[:5]:
    print(f'  {f}')

# 단일 진실 테이블 존재 여부 (이전 실행 결과)
truth_tables = {
    'erangel_features.parquet':  '01 생성',
    'erangel_clustered.parquet': '02 생성',
    'miramar_clustered.parquet': '03 생성',
    'taego_clustered.parquet':   '03 생성',
    'rondo_clustered.parquet':   '03 생성',
}
print('\n테이블 현황:')
for fname, src in truth_tables.items():
    path = os.path.join(output_dir, fname)
    exists = os.path.exists(path)
    print(f'  {"" if exists else ""} {fname:<40} ({src})')


 config.py 확인: C:\배그분석\config.py

analysis_output 폴더: 존재

텔레메트리 parquet: 3463개
  C:\배그분석\temp_parquet_files_kakao_20260212\Erangel_01a4a247-7e5a-4835-a2f7-a3f7c9fb2aac.parquet
  C:\배그분석\temp_parquet_files_kakao_20260212\Erangel_01a58e2c-d5b5-4741-9c2b-5ca94b6873fb.parquet
  C:\배그분석\temp_parquet_files_kakao_20260212\Erangel_046d6575-7184-4e61-bbc9-6ba9b7e30339.parquet
  C:\배그분석\temp_parquet_files_kakao_20260212\Erangel_06b59825-68cc-444a-a4bf-70df6a06bf01.parquet
  C:\배그분석\temp_parquet_files_kakao_20260212\Erangel_07b98f57-f539-4dd6-9499-138531d35d34.parquet
  외 3458개

matches CSV: 28개
  C:\배그분석\matches_kakao_20260212.csv
  C:\배그분석\matches_kakao_20260213.csv
  C:\배그분석\matches_kakao_20260214.csv
  C:\배그분석\matches_kakao_20260215.csv
  C:\배그분석\matches_kakao_20260216.csv

테이블 현황:
   erangel_features.parquet                 (01 생성)
   erangel_clustered.parquet                (02 생성)
   miramar_clustered.parquet                (03 생성)
   taego_clustered.parquet                  (03 생성)
   ron